MERGE ACTIVITY 2 WITH ACTIVITY 1

In [ ]:
# ============================================================
# ACTIVITY 2: ENHANCED SVM USING OPTUNA
# Automated Hyperparameter Tuning
# ============================================================

!pip install optuna -q

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

In [ ]:
# ============================================================
# 1. Define Optuna objective function
# ============================================================

def objective(trial):

    # ENHANCEMENT 1: Tune kernel type
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])

    # ENHANCEMENT 2: Tune C value
    C = trial.suggest_float("C", 0.01, 100, log=True)

    # ENHANCEMENT 3: Tune gamma for RBF kernel
    if kernel == "rbf":
        gamma = trial.suggest_float("gamma", 0.0001, 1, log=True)
    else:
        gamma = "scale"

    # ENHANCEMENT 4: Tune class weight to improve churn detection
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    enhanced_svm = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(
            kernel=kernel,
            C=C,
            gamma=gamma,
            class_weight=class_weight
        ))
    ])

    # ENHANCEMENT 5: Use cross-validation and F1-score
    score = cross_val_score(
        enhanced_svm,
        X_train,
        y_train,
        cv=3,
        scoring="f1"
    ).mean()

    return score

In [ ]:
# ============================================================
# 2. Run Optuna automated search
# ============================================================

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best F1 Score from Optuna:", study.best_value)
print("Best Parameters:")
print(study.best_params)

In [ ]:
# ============================================================
# 3. Build final enhanced SVM using best parameters
# ============================================================

best_params = study.best_params

enhanced_svm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC(
        kernel=best_params["kernel"],
        C=best_params["C"],
        gamma=best_params.get("gamma", "scale"),
        class_weight=best_params["class_weight"]
    ))
])

enhanced_svm.fit(X_train, y_train)

enhanced_pred = enhanced_svm.predict(X_test)

In [ ]:
# ============================================================
# 4. Evaluate enhanced SVM model
# ============================================================

enhanced_accuracy = accuracy_score(y_test, enhanced_pred)
enhanced_precision = precision_score(y_test, enhanced_pred)
enhanced_recall = recall_score(y_test, enhanced_pred)
enhanced_f1 = f1_score(y_test, enhanced_pred)

print("===== OPTUNA ENHANCED SVM MODEL PERFORMANCE =====")
print("Accuracy :", round(enhanced_accuracy, 4))
print("Precision:", round(enhanced_precision, 4))
print("Recall   :", round(enhanced_recall, 4))
print("F1 Score :", round(enhanced_f1, 4))

In [ ]:
# ============================================================
# 5. Compare Baseline SVM vs Optuna Enhanced SVM
# ============================================================

comparison = pd.DataFrame({
    "Model": ["Baseline SVM", "Optuna Enhanced SVM"],
    "Accuracy": [accuracy, enhanced_accuracy],
    "Precision": [precision, enhanced_precision],
    "Recall": [recall, enhanced_recall],
    "F1 Score": [f1, enhanced_f1]
})

comparison